# Model selection is an estimation and governance problem

**Question.** Which candidate should advance, given predictive evidence, uncertainty, resource constraints, and the risk of repeated search?

This report deliberately retains every trusted reference candidate. Its single split is an instructional smoke test—not a final selection procedure. A project must design validation around its observation dependence and keep final-test evidence sealed until the procedure is frozen.

In [ ]:
import matplotlib.pyplot as plt

from dsm_project.config import load_config
from dsm_project.contracts import TaskKind
from dsm_project.data import load_project_dataset
from dsm_project.evaluation import comparison_table
from dsm_project.experiments import available_models

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
models = available_models(TaskKind.CLASSIFICATION)

table = comparison_table(
    dataset,
    config,
    task_name="classification",
    model_names=models,
).sort_values("value", ascending=False)
display(table)

## Start with task meaning and a baseline

Balanced accuracy averages recall across true classes. It is useful when each class deserves equal weight, but it may not match operational costs. A majority prediction is a simple sanity check; a domain rule can be a stronger baseline. Selection must be written in terms of the user's loss, not whichever metric makes a preferred model look best.

In [ ]:
colors = ["#1b9e77" if index == 0 else "#7570b3" for index in range(len(table))]
table.sort_values("value").plot.barh(
    x="model",
    y="value",
    legend=False,
    color=list(reversed(colors)),
    figsize=(9, 5),
)
plt.xlabel("balanced accuracy on instructional holdout")
plt.ylabel("")
plt.title("Retain the full candidate record")
plt.xlim(0, 1.05)
plt.tight_layout();

print("Instructional leader:", table.iloc[0]["model"])
print("This is not yet an approved production model.")

## A defensible selection design

Before opening final-test data, define:

1. the unit of independence and whether groups, space or time constrain splits;
2. preprocessing and hyperparameter search inside each training fold;
3. baselines and eligible model families;
4. primary and secondary metrics, uncertainty summaries and important slices;
5. resource, latency, interpretability and safety gates;
6. a tie or practical-equivalence rule; and
7. the condition under which no model is promoted.

Repeatedly choosing the best result on the same holdout overfits the holdout. Nested resampling can separate hyperparameter selection from performance estimation when data and computation permit.

## Promotion is a software event

A selected experiment is not yet a product. Promotion should create a versioned artifact containing model identity, feature order, configuration, data provenance, evidence, source revision and limitations. Integration and API tests must pass against that exact artifact. Monitoring and rollback must exist before deployment.

The winner table remains useful because it preserves losing candidates and makes later regressions visible. Do not delete evidence merely because a candidate was not selected.

## Student deliverable

Replace this smoke-test table with a project-owned comparison module and notebook that explains the validation design, uncertainty, slices and resource evidence. Include the registered student implementations and clearly labeled trusted references. Freeze the promotion rule in a decision record, obtain review, evaluate the final test once, complete the model card, and record why the selected model—or a decision not to deploy—follows from the evidence.